# JME Call for Papers Revision

In [1]:
import glob, os, json, time, random, pickle, math, re
import pandas as pd
import numpy as np
import torch

import matplotlib.pyplot as plt
import cv2, PIL, base64
from moviepy import VideoFileClip

from tqdm import tqdm, trange
from functools import partial
from pathlib import Path
from collections import defaultdict

import utils, data_utils

os.environ['KMP_DUPLICATE_LIB_OK']='True'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Additional Data Collection

In [9]:
url = "https://www.youtube.com/watch?v=dGNAwxi9uwA"
download_dir = "data/youtube"
# data_utils.download_video(url, download_dir, download_sections=(12,21))
data_utils.download_video(url, download_dir)

[youtube] Extracting URL: https://www.youtube.com/watch?v=dGNAwxi9uwA
[youtube] dGNAwxi9uwA: Downloading webpage


[youtube] dGNAwxi9uwA: Downloading visionos player API JSON
[youtube] dGNAwxi9uwA: Downloading m3u8 information
[info] dGNAwxi9uwA: Downloading 1 format(s): 616
[hlsnative] Downloading m3u8 manifest
[hlsnative] Total fragments: 99
[download] Destination: data\youtube\How_To_Lay_Out_Walls_Floors_and_Roofs__Explaining_Layout.mp4
[download] 100% of  283.04MiB in 00:00:37 at 7.50MiB/s                  


'data/youtube\\How_To_Lay_Out_Walls_Floors_and_Roofs__Explaining_Layout.mp4'

In [10]:
source_dir = "data/youtube"
target_dir = "data/new_clipped"

source_video_filename = "How_To_Lay_Out_Walls_Floors_and_Roofs__Explaining_Layout" + ".mp4"
source_video_path = os.path.join(source_dir, source_video_filename)

start_time = 2*60 + 26
end_time = 3*60 + 6
target_video_path = os.path.join(target_dir, f"clipped_{start_time}_{end_time}_{source_video_filename}")

data_utils.clip_video(source_video_path, start_time, end_time, target_video_path)

MoviePy - Building video data/new_clipped\clipped_146_186_How_To_Lay_Out_Walls_Floors_and_Roofs__Explaining_Layout.mp4.
MoviePy - Writing video data/new_clipped\clipped_146_186_How_To_Lay_Out_Walls_Floors_and_Roofs__Explaining_Layout.mp4



MoviePy - Done !
MoviePy - video ready data/new_clipped\clipped_146_186_How_To_Lay_Out_Walls_Floors_and_Roofs__Explaining_Layout.mp4


In [12]:
video_paths = "data/new_clipped/*.mp4"
video_paths = glob.glob(video_paths)
for video_path in video_paths:
    v_path = Path(video_path)
    data_utils.extract_frames(v_path, target_dir_fps1 = Path("data/new_frames_fps1"), target_dir_fps8 = Path("data/new_frames_fps8"))

In [18]:
frame1_dir = "data/new_frames_fps1"
frame_paths = glob.glob(os.path.join(frame1_dir, "*.jpg"))

def sort_key(path):
    filename = os.path.splitext(os.path.basename(path))[0]
    parts = filename.rsplit("_", 2)

    base_filename = parts[0]
    second = int(parts[1])
    frame_idx = int(parts[2])

    return base_filename, second, frame_idx

frame_paths = sorted(frame_paths, key=sort_key)

base_filenames = []
seconds = []
for frame_path in frame_paths:
    base_filename = os.path.basename(frame_path).replace(".jpg", "")
    second = base_filename.split("_")[-2]
    base_filenames.append(base_filename)
    seconds.append(second)

new_df = pd.DataFrame({"base_filename": base_filenames, "second": seconds})
new_df.to_csv("data/new_GT_annotated.csv", index=False)

## Dataset Statistics

In [34]:
import pandas as pd

# df = pd.read_csv("data/GT_fully_annotated.csv")
# df = pd.read_csv("data/new_GT_annotated.csv")
df = pd.read_csv("data/GT_merged.csv")

ignore_basefilenames = [
    "clipped_1_14_Semiautomatic_nail_gun_accidents",
    "clipped_0_13_overhead_drilling_v1",
    "clipped_0_15_Drill work for roof ceiling shorts viral youtubeshortvideo roof",
    "clipped_0_22_fall ceiling drilling videoshot",
    "clipped_165_203_How to FRAME a Wall - 3 EASY STEPS_1080p",
    "clipped_0_19_Hand taping inside corners is always fun! LEVEL5 knifes always make the job easier",
    "clipped_0_25_Finishing Drywall Butt Joints with LEVEL5 Hand Tools",
    "clipped_0_11_Hard working Malaysia constructionworker",
    "clipped_0_12_Amazing fastest work rebar tying skill",
    "clipped_0_15_GUARANTEE",
    "clipped_11_25_Rebar tying (2)"
]

df = df[~df["base_filename"].isin(ignore_basefilenames)].reset_index(drop=True)

annotation_cols = ["action_1", "action_2", "action_3"]
annotation_cols = [col for col in annotation_cols if col in df.columns]

all_annotations = (
    df[annotation_cols]
    .stack()                  # flatten the three columns into one
    .astype(str)
    .str.strip()             # remove leading/trailing whitespace
)

# remove empty strings / nan
all_annotations = all_annotations[
    (all_annotations != "") &
    (all_annotations.str.lower() != "nan")
]

# count occurrences
annotation_counts = all_annotations.value_counts()

# print results
print("=== Kinds of Annotations ===")
print(annotation_counts)

print(f"\nTotal number of unique annotations: {annotation_counts.shape[0]}")

num_videos = df["base_filename"].nunique()
print(f"Number of unique videos: {num_videos}") #NOTE: Why 489? not 100?
num_rows = len(df)
print(f"Number of rows: {num_rows}")

=== Kinds of Annotations ===
none                              1251
lay brick/block                    336
drill                              315
plaster                            278
tie rebar                          249
lift/carry sheets                  243
apply adhesive/mortar              205
install tiles                      194
lift/carry window                  168
install roofing sheets             164
lift/carry cement bags             158
climb/climb down ladders/steps     133
screed                             126
attach rigging                     105
lift/carry door                     91
drive compactor                     85
measure dimensions/distances        85
weld                                80
paint                               73
hammer                              67
drive vehicles                      66
lift/carry hose                     60
measure the level                   59
push/pull cart                      55
sweep                              

In [32]:
action_cols = ["action_1", "action_2", "action_3"]

unique_action_counts = (
    df.melt(
        id_vars=["base_filename"],
        value_vars=action_cols,
        value_name="action"
    )
    .dropna(subset=["action"])
    .groupby("base_filename")["action"]
    .nunique()
    .reset_index(name="num_unique_actions")
)

avg_unique_actions = unique_action_counts["num_unique_actions"].mean()

print(f"Average number of unique actions per video: {avg_unique_actions:.2f}")

Average number of unique actions per video: 2.44


In [ ]:
# df1 = pd.read_csv("data/GT_fully_annotated.csv")
df1 = pd.read_csv("data/GT_fully_annotated_jinsik.csv")
df2 = pd.read_csv("data/GT_fully_annotated_seongju.csv")
# 건우-성주 98.33 건우-진식 89.09 진식-성주 89.09

action_cols = ["action_1", "action_2", "action_3"]

# Match rows by video and second
merged = pd.merge(
    df1,
    df2,
    on=["base_filename", "second"],
    suffixes=("_rater1", "_rater2"),
    how="inner"
)


def get_action_set(row, suffix):
    actions = {
        row[f"{col}_{suffix}"]
        for col in action_cols
        if pd.notna(row[f"{col}_{suffix}"])
    }
    
    # Remove whitespace just in case
    return {str(action).strip() for action in actions}


# def check_agreement(row):
#     actions_1 = get_action_set(row, "rater1")
#     actions_2 = get_action_set(row, "rater2")

#     # Agree if at least one action overlaps
#     return int(len(actions_1 & actions_2) > 0)

def check_agreement(row):
    actions_1 = get_action_set(row, "rater1")
    actions_2 = get_action_set(row, "rater2")

    # Agree only when the entire action sets are identical
    return int(actions_1 == actions_2)


merged["agree"] = merged.apply(check_agreement, axis=1)


# Agreement ratio
agreement_ratio = merged["agree"].mean()

print(f"Matched rows: {len(merged)}")
print(f"Agreed rows: {merged['agree'].sum()}")
print(f"Not agreed rows: {(merged['agree'] == 0).sum()}")
print(f"Agreement ratio: {agreement_ratio:.4f}")
print(f"Agreement percentage: {agreement_ratio * 100:.2f}%")
avg_agreement_ratio = (89.09*2+98.33)/3
print(f"Avg agreement ratio: {avg_agreement_ratio:.2f}%")

Matched rows: 4309
Agreed rows: 3839
Not agreed rows: 470
Agreement ratio: 0.8909
Agreement percentage: 89.09%


In [43]:
(89.09*2+98.33)/3

92.17

In [8]:
df = pd.read_csv("data/GT_fully_annotated.csv")
action_cols = ["action_1", "action_2", "action_3"]

# Convert action columns from wide format to long format
actions = df.melt(
    id_vars=["base_filename"],
    value_vars=action_cols,
    value_name="action_class"
)

# Remove empty annotations
actions = actions.dropna(subset=["action_class"])

# Each action is counted only once per video
actions_unique = actions.drop_duplicates(subset=["base_filename", "action_class"])

# Count the number of unique videos containing each action
action_counts = (
    actions_unique
    .groupby("action_class")["base_filename"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="video_count")
)

print(action_counts)

                      action_class  video_count
0                             none           84
1                            drill           19
2   climb/climb down ladders/steps           16
3            apply adhesive/mortar           12
4                lift/carry window           12
5                lift/carry sheets            9
6                        tie rebar            8
7                  lay brick/block            8
8                          plaster            7
9                           screed            6
10                   install tiles            5
11                  push/pull cart            5
12                           paint            5
13    measure dimensions/distances            4
14                          hammer            4
15          install roofing sheets            4
16                  attach rigging            3
17                           sweep            3
18                 drive compactor            3
19          lift/carry cement bags      

In [39]:
df = pd.read_csv("data/GT_unique_basefilenames.csv")
# ratio of value in "uniformat"
df["uniformat"].value_counts(normalize=True).sort_values(ascending=False)

uniformat
shell                0.38
interiors            0.34
substructure         0.20
building sitework    0.08
Name: proportion, dtype: float64

## Gemini to GPT

In [6]:
gemini_path = "output/inference_results_gemini-3.1-pro-preview_videos.json"
# gemini_path = "output/inference_results_gemini-3.5-flash-lite_videos.json"
org_gemini_filename = os.path.basename(gemini_path).replace(".json", "")

gpt_path = "output/inference_results_gpt-5.6-luna.json"
out_path = f"output/{org_gemini_filename}_gpt_format.json"
# out_path = "output/inference_results_gemini-3.1-flash-lite-preview_video_gpt_format.json"

gemini_rows = utils.read_jsonl(gemini_path)
gpt_rows = utils.read_jsonl(gpt_path)

gemini_by_key = {
    data_utils.video_key_from_video_path(r["video_path"]): r
    for r in gemini_rows
}
gpt_by_key = defaultdict(list)

for r in gpt_rows:
    key, sec, frame_idx = data_utils.parse_frame_path(r["frame_path"])
    rr = dict(r)
    rr["_video_key"] = key
    rr["_second"] = sec
    rr["_frame_idx"] = frame_idx
    gpt_by_key[key].append(rr)

converted = []
missing_keys = set()

for key, frames in gpt_by_key.items():
    gem = gemini_by_key.get(key)
    if gem is None:
        missing_keys.add(key)
        continue

    n = len(frames)
    per_frame_cost = gem.get("inference_cost", 0) / n if n > 0 else 0
    per_frame_time = gem.get("inference_time", 0) / n if n > 0 else 0

    for fr in sorted(frames, key=lambda x: (x["_second"], x["_frame_idx"])):
        converted.append({
            "frame_path": fr["frame_path"],
            "action": data_utils.action_at_second(gem.get("segments", []), fr["_second"]),
            "inference_cost": per_frame_cost,
            "inference_time": per_frame_time,
        })

with open(out_path, "w", encoding="utf-8") as f:
    for row in converted:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("gemini videos:", len(gemini_rows))
print("gpt frames:", len(gpt_rows))
print("converted frames:", len(converted))
print("missing video keys:", len(missing_keys))
print(sorted(list(missing_keys))[:10])
print("output:", out_path)
print(converted[:3])

gemini videos: 100
gpt frames: 4517
converted frames: 4517
missing video keys: 0
[]
output: output/inference_results_gemini-3.1-pro-preview_videos_gpt_format.json
[{'frame_path': 'data/frames_fps1\\clipped_0_10_NYC_crane_and__rigging_rebar_bundles_team_safety_always_firstcomment_if_you_do_construction_0_17.jpg', 'action': 'attach rigging', 'inference_cost': 0.00039755555555555556, 'inference_time': 0.9886753294203017}, {'frame_path': 'data/frames_fps1\\clipped_0_10_NYC_crane_and__rigging_rebar_bundles_team_safety_always_firstcomment_if_you_do_construction_1_47.jpg', 'action': 'attach rigging', 'inference_cost': 0.00039755555555555556, 'inference_time': 0.9886753294203017}, {'frame_path': 'data/frames_fps1\\clipped_0_10_NYC_crane_and__rigging_rebar_bundles_team_safety_always_firstcomment_if_you_do_construction_2_77.jpg', 'action': 'attach rigging', 'inference_cost': 0.00039755555555555556, 'inference_time': 0.9886753294203017}]


## Statistic Analsysis

In [7]:
import numpy as np
from scipy import stats
from statsmodels.stats.oneway import anova_oneway

# 각 모델별 3회 결과
# 순서:
# Acc, Foreground_Acc, Overlap_Acc, F1, one_min_time, one_min_cost

data = {
    "gpt-5.6-terra": [
        [0.7835, 0.8169, 0.8337, 0.7357, 103.1044, 0.2981],
        [0.7833, 0.8211, 0.8322, 0.7375, 115.5911, 0.2991],
        [0.7815, 0.8172, 0.8320, 0.7386, 122.8786, 0.2993],
    ],

    "gpt-5.6-luna": [
        [0.7658, 0.8105, 0.8118, 0.7201, 143.3120, 0.0329],
        [0.7605, 0.8038, 0.8061, 0.7148, 154.1766, 0.0330],
        [0.7806, 0.8247, 0.8273, 0.7475, 150.4988, 0.0329],
    ],
    "claude-sonnet-5": [
    [0.7660, 0.8548, 0.8198, 0.7513, 150.8571, 0.3486],
    [0.7627, 0.8514, 0.8158, 0.7576, 142.5471, 0.3487],
    [0.7866, 0.8914, 0.8408, 0.8733, 149.7273, 0.3492],
    ],
    
    "claude-haiku-4-5": [
        [0.5588, 0.6351, 0.6092, 0.5494, 86.2583, 0.1248],
        [0.5776, 0.6610, 0.6296, 0.5451, 55.7697, 0.1260],
        [0.5721, 0.6598, 0.6221, 0.5937, 56.4651, 0.1262],
    ],
    
    "gemini-3.1-pro-preview": [
        [0.7483, 0.7729, 0.8145, 0.6993, 43.1715, 0.0144],
        [0.7361, 0.7577, 0.8003, 0.6859, 28.8264, 0.0144],
        [0.8019, 0.8317, 0.8709, 0.8160, 28.8440, 0.0143],
    ],

    "gemini-3.5-flash-lite": [
        [0.7342, 0.7978, 0.8123, 0.7091, 21.1381, 0.0022],
        [0.7516, 0.8132, 0.8193, 0.7227, 19.7562, 0.0023],
        [0.7532, 0.8344, 0.8242, 0.8102, 10.6108, 0.0023],
    ],
}

metrics = [
    "Accuracy",
    "Foreground Accuracy",
    "Overlap Accuracy",
    "Macro F1",
    "1-min inference time",
    "1-min inference cost",
]


# ------------------------------------------------------
# 1. Mean, variability (SD), and 95% confidence interval
# ------------------------------------------------------

print("\n===== Variability and 95% CI =====\n")

for metric_idx, metric in enumerate(metrics):

    print(f"\n--- {metric} ---")

    for model, values in data.items():

        x = np.array(values)[:, metric_idx]

        n = len(x)
        mean = np.mean(x)

        # Sample standard deviation
        sd = np.std(x, ddof=1)

        # Standard error
        se = sd / np.sqrt(n)

        # t-based 95% CI because n = 3
        t_crit = stats.t.ppf(0.975, df=n-1)

        ci_low = mean - t_crit * se
        ci_high = mean + t_crit * se

        print(
            f"{model:25s} "
            f"Mean = {mean:.4f}, "
            f"SD = {sd:.4f}, "
            f"95% CI = [{ci_low:.4f}, {ci_high:.4f}]"
        )


# ------------------------------------------------------
# 2. Welch one-way ANOVA
# ------------------------------------------------------

print("\n\n===== Welch ANOVA =====\n")

for metric_idx, metric in enumerate(metrics):

    groups = [
        np.array(values)[:, metric_idx]
        for values in data.values()
    ]

    # use_var='unequal' = Welch's ANOVA
    result = anova_oneway(
        groups,
        use_var="unequal"
    )

    print(f"\n{metric}")

    print(
        f"Welch F({result.df_num:.3f}, "
        f"{result.df_denom:.3f}) "
        f"= {result.statistic:.4f}"
    )

    print(f"p = {result.pvalue:.8f}")

    if result.pvalue < 0.05:
        print("→ Significant difference among model means.")
    else:
        print("→ No statistically significant difference.")


===== Variability and 95% CI =====


--- Accuracy ---
gpt-5.6-terra             Mean = 0.7828, SD = 0.0011, 95% CI = [0.7800, 0.7855]
gpt-5.6-luna              Mean = 0.7690, SD = 0.0104, 95% CI = [0.7431, 0.7948]
claude-sonnet-5           Mean = 0.7718, SD = 0.0130, 95% CI = [0.7396, 0.8039]
claude-haiku-4-5          Mean = 0.5695, SD = 0.0097, 95% CI = [0.5455, 0.5935]
gemini-3.1-pro-preview    Mean = 0.7621, SD = 0.0350, 95% CI = [0.6751, 0.8491]
gemini-3.5-flash-lite     Mean = 0.7463, SD = 0.0105, 95% CI = [0.7202, 0.7725]

--- Foreground Accuracy ---
gpt-5.6-terra             Mean = 0.8184, SD = 0.0023, 95% CI = [0.8126, 0.8242]
gpt-5.6-luna              Mean = 0.8130, SD = 0.0107, 95% CI = [0.7865, 0.8395]
claude-sonnet-5           Mean = 0.8659, SD = 0.0222, 95% CI = [0.8108, 0.9210]
claude-haiku-4-5          Mean = 0.6520, SD = 0.0146, 95% CI = [0.6157, 0.6883]
gemini-3.1-pro-preview    Mean = 0.7874, SD = 0.0391, 95% CI = [0.6903, 0.8845]
gemini-3.5-flash-lite     Mean = 0.8

In [30]:
from scipy import stats


def compare_models(df, model1, model2, metric, alpha=0.05):
    """
    Compare two models for one metric using Welch's t-test.

    Parameters
    ----------
    df : pandas.DataFrame
        Must contain 'Models' and metric columns.
    model1 : str
        First model name.
    model2 : str
        Second model name.
    metric : str
        Metric column name.
    alpha : float
        Significance level. Default = 0.05.

    Returns
    -------
    dict
        Means, t-statistic, p-value, and conclusion.
    """

    # Remove precomputed average rows
    data = df[
        ~df["Models"].str.contains(r"\(avg\)", regex=True)
    ].copy()

    models = data["Models"].unique()

    if model1 not in models:
        raise ValueError(f"Model not found: {model1}")
    if model2 not in models:
        raise ValueError(f"Model not found: {model2}")
    if metric not in data.columns:
        raise ValueError(f"Metric not found: {metric}")

    # Get individual runs
    x = data.loc[
        data["Models"] == model1, metric
    ].dropna().values

    y = data.loc[
        data["Models"] == model2, metric
    ].dropna().values

    # Welch's t-test
    t_stat, p_value = stats.ttest_ind(
        x,
        y,
        equal_var=False
    )

    mean_model1 = x.mean()
    mean_model2 = y.mean()

    # Conclusion
    if p_value < alpha:
        if mean_model1 > mean_model2:
            conclusion = (
                f"Significant difference: {model1} has a higher mean "
                f"than {model2} (p = {p_value:.4g})."
            )
        else:
            conclusion = (
                f"Significant difference: {model2} has a higher mean "
                f"than {model1} (p = {p_value:.4g})."
            )
    else:
        conclusion = (
            f"No statistically significant difference between "
            f"{model1} and {model2} (p = {p_value:.4g})."
        )

    return {
        "metric": metric,
        "model1": model1,
        "model2": model2,
        "mean_model1": mean_model1,
        "mean_model2": mean_model2,
        "t_statistic": t_stat,
        "p_value": p_value,
        "significant": bool(p_value < alpha),
        "conclusion": conclusion
    }

In [37]:
df = pd.read_excel("output/Revision_result.xlsx")
metric = "Acc"
result1 = compare_models(df, "gpt-5.6-terra", "gpt-5.6-luna", metric)
result2 = compare_models(df, "claude-sonnet-5", "claude-haiku-4-5", metric)
result3 = compare_models(df, "gemini-3.1-pro-preview", "gemini-3.5-flash-lite", metric)
print(f"GPT Terra vs GPT Luna: {result1['conclusion']}")
print(f"Claude Sonnet 5 vs Claude Haiku 4-5: {result2['conclusion']}")
print(f"Gemini 3.1 Pro Preview vs Gemini 3.5 Flash Lite: {result3['conclusion']}")

GPT Terra vs GPT Luna: No statistically significant difference between gpt-5.6-terra and gpt-5.6-luna (p = 0.1473).
Claude Sonnet 5 vs Claude Haiku 4-5: Significant difference: claude-sonnet-5 has a higher mean than claude-haiku-4-5 (p = 4.911e-05).
Gemini 3.1 Pro Preview vs Gemini 3.5 Flash Lite: No statistically significant difference between gemini-3.1-pro-preview and gemini-3.5-flash-lite (p = 0.5224).


## Qwen (Opensource)

In [44]:
from openai import OpenAI
import os, json

API_KEY_JSON_PATH = "APIKEY/api_key.json"
API_KEY_JSON = json.load(open(API_KEY_JSON_PATH, "r"))
DASHSCOPE_API_KEY = API_KEY_JSON["Qwen_yong"]

client = OpenAI(
    # If the environment variable is not set, replace it with your Model Studio API key: api_key="sk-xxx"
    # api_key=os.getenv("DASHSCOPE_API_KEY"),
    api_key=DASHSCOPE_API_KEY,
    # base_url="https://dashscope-intl.aliyuncs.com/compatible-mode/v1",
    base_url="https://ws-6t89kzdkpqlr04f7.us-east-1.maas.aliyuncs.com/compatible-mode/v1"
)

messages = [{"role": "user", "content": "Who are you"}]
completion = client.chat.completions.create(
    model="qwen3.8-max",  # You can replace this with another deep thinking models
    messages=messages,
    extra_body={"enable_thinking": True},
    stream=True
)
is_answering = False  # Indicates whether the response phase has started
print("\n" + "=" * 20 + "Thinking process" + "=" * 20)
for chunk in completion:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta
    if hasattr(delta, "reasoning_content") and delta.reasoning_content is not None:
        if not is_answering:
            print(delta.reasoning_content, end="", flush=True)
    if hasattr(delta, "content") and delta.content:
        if not is_answering:
            print("\n" + "=" * 20 + "Full response" + "=" * 20)
            is_answering = True
        print(delta.content, end="", flush=True)

PermissionDeniedError: Error code: 403 - {'error': {'message': 'Access to model denied. Please make sure you are eligible for using the model.', 'id': 'c4b6d452-f411-44b7-af13-814733b06d5f', 'type': 'AccessDenied.Unpurchased', 'code': 'AccessDenied.Unpurchased'}}

In [33]:
from openai import OpenAI
import os, json
import pandas as pd

import utils
from genais import StructuredResponse

API_KEY_JSON_PATH = "APIKEY/api_key.json"
API_KEY_JSON = json.load(open(API_KEY_JSON_PATH, "r"))
OLLAMA_API_KEY = API_KEY_JSON["Ollama_yong"]

df_class_path = "data/action_classes.csv"
df_class = pd.read_csv(df_class_path)
action_classes = df_class["class"].tolist()

if "none" not in action_classes:
    action_classes.append("none")

BASIC_PROMPT = f"""
# ROLE:
You are an expert construction manager analyzing video frames from construction site videos.

# INSTRUCTION:
Determine the worker's current action from the provided frame or frames.

An action should be identified ONLY when the worker is actively performing a construction task through direct physical interaction with relevant work entities, such as:
- tools
- materials
- construction equipment
- building components

Direct physical interaction means the worker's hand(s) are visibly in contact with the object(s) involved in the task.

Return "none" in the following cases:
- the worker is preparing to start a task
- the worker is observing, waiting, or idle
- the worker's hands are not in active contact with relevant work objects
- the visible action does not clearly match any provided action options
- the worker is performing a non-target action outside the provided action list

Choose ONLY from the provided action options below:
{action_classes}

Be conservative. If the action is ambiguous or insufficiently visible, return "none".

# OUTPUT FORMAT:
Return a valid JSON object only.
Key: "action"
Value: One action from the provided choices or "none"

# EXAMPLE:
Input: A frame showing a worker using a hammer to drive a nail into a piece of wood
Output: "action": "hammer"
"""

client = OpenAI(
    base_url='http://localhost:11434/v1/',
    api_key=OLLAMA_API_KEY,  # required but ignored
)
frame_path = "data/frames_fps1/clipped_0_9_broom_8_257.jpg"
base64_frame = utils.encode_image(frame_path)

# qwen_input = [
#         {
#             "role": "user",
#             "content": [
#                 { "type": "text", "text": BASIC_PROMPT},
#                 {
#                     "type": "image_url",
#                     "image_url": f"data:image/jpeg;base64,{base64_frame}",
#                     # "detail": "auto"
#                 },
#             ],
#         }
#     ]

qwen_input = [
                {
                    "role": "user",
                    "content": [
                        { "type": "input_text", "text": BASIC_PROMPT },
                        {
                            "type": "input_image",
                            "image_url": f"data:image/jpeg;base64,{base64_frame}",
                            "detail": "auto"
                        },
                    ],
                }
            ]

# response = client.chat.completions.create(
response = client.responses.parse(
    model='qwen3.8:27b',
    # messages=qwen_input,
    input=qwen_input,
    # max_tokens=4096,
    reasoning={"effort": "low"},
    text_format=StructuredResponse
)
print(response.output_parsed.action[0])

screed


In [23]:
help(client.responses.parse)

Help on method parse in module openai.resources.responses.responses:

parse(*, text_format: 'type[TextFormatT] | Omit' = <openai.Omit object at 0x000001881BACA780>, background: 'Optional[bool] | Omit' = <openai.Omit object at 0x000001881BACA780>, context_management: 'Optional[Iterable[response_create_params.ContextManagement]] | Omit' = <openai.Omit object at 0x000001881BACA780>, conversation: 'Optional[response_create_params.Conversation] | Omit' = <openai.Omit object at 0x000001881BACA780>, include: 'Optional[List[ResponseIncludable]] | Omit' = <openai.Omit object at 0x000001881BACA780>, input: 'Union[str, ResponseInputParam] | Omit' = <openai.Omit object at 0x000001881BACA780>, instructions: 'Optional[str] | Omit' = <openai.Omit object at 0x000001881BACA780>, max_output_tokens: 'Optional[int] | Omit' = <openai.Omit object at 0x000001881BACA780>, max_tool_calls: 'Optional[int] | Omit' = <openai.Omit object at 0x000001881BACA780>, metadata: 'Optional[Metadata] | Omit' = <openai.Omit o

## Figure

In [26]:
import plotly.graph_objects as go


SHOW_TICKS = True      # True = 축 숫자 ON / False = OFF
SHOW_LABELS = False     # True = point label ON / False = OFF

models = [
    "GPT Terra",
    "GPT Luna",
    "Claude Sonnet 5",
    "Claude Haiku 4-5",
    "Gemini 3.1 Pro Preview",
    "Gemini 3.5 Flash Lite"
]

cost = [0.2988, 0.0329, 0.3488, 0.1257, 0.0144, 0.0023]
time = [113.8580, 149.3291, 147.7105, 66.1644, 33.6140, 17.1684]
accuracy = [0.7828, 0.7690, 0.7718, 0.5695, 0.7621, 0.7463]

colors = [
    "green",   # GPT Terra
    "green",   # GPT Luna
    "red",     # Claude Sonnet 5
    "red",     # Claude Haiku 4-5
    "blue",    # Gemini 3.1 Pro Preview
    "blue"     # Gemini 3.5 Flash Lite
]

symbols = [
    "circle",       # GPT Terra
    "circle",       # GPT Luna
    "cross",        # Claude Sonnet 5
    "cross",        # Claude Haiku 4-5
    "diamond",      # Gemini 3.1 Pro Preview
    "diamond"       # Gemini 3.5 Flash Lite
]

x_min = 0.0
x_max = 0.40

y_min = 0.0
y_max = 160

z_min = 0.50
z_max = 1

fig = go.Figure()

for x, y, z in zip(cost, time, accuracy):

    # --------------------------------------------------------
    # Vertical projection: point -> bottom plane
    # --------------------------------------------------------
    fig.add_trace(
        go.Scatter3d(
            x=[x, x],
            y=[y, y],
            z=[z_min, z],

            mode="lines",

            line=dict(
                color="gray",
                width=4,
                dash="dot"
            ),

            hoverinfo="skip",
            showlegend=False
        )
    )


    # --------------------------------------------------------
    # Projection toward Y wall
    # point -> y = y_min
    # --------------------------------------------------------
    fig.add_trace(
        go.Scatter3d(
            x=[x, x],
            y=[y_min, y],
            z=[z, z],

            mode="lines",

            line=dict(
                color="gray",
                width=4,
                dash="dot"
            ),

            hoverinfo="skip",
            showlegend=False
        )
    )


    # --------------------------------------------------------
    # Projection toward X wall
    # point -> x = x_min
    # --------------------------------------------------------
    fig.add_trace(
        go.Scatter3d(
            x=[x_min, x],
            y=[y, y],
            z=[z, z],

            mode="lines",

            line=dict(
                color="gray",
                width=4,
                dash="dot"
            ),

            hoverinfo="skip",
            showlegend=False
        )
    )

fig.add_trace(
    go.Scatter3d(x=cost, y=time, z=accuracy,
        mode="markers",
        # mode="markers+text",   # <-- Point labels ON

        text=models,
        textposition="top center",

        marker=dict(size=12, color=colors, symbol=symbols, opacity=1.0),

        hovertemplate=(
            "<b>%{text}</b><br>"
            "Cost: %{x:.3f}<br>"
            "Time: %{y:.1f}<br>"
            "Accuracy: %{z:.4f}"
            "<extra></extra>"
        )
    )
)


fig.update_layout(width=900, height=700,
    margin=dict(l=0, r=0, b=0, t=0),
    showlegend=False,
    scene=dict(
        xaxis=dict(
            title="",             # no axis label
            showticklabels=SHOW_TICKS, # no numbers
            ticks="",
            showgrid=True,
            zeroline=False
        ),

        yaxis=dict(
            title="",
            showticklabels=SHOW_TICKS,
            ticks="",
            showgrid=True,
            zeroline=False
        ),

        zaxis=dict(
            title="",
            showticklabels=SHOW_TICKS,
            ticks="",
            showgrid=True,
            zeroline=False,
            tick0=0.5,
            dtick=0.1,
            range=[z_min, z_max]
        ),

        # Initial camera angle
        camera=dict(
            eye=dict(
                x=1.5,
                y=1.5,
                z=1.2
            )
        ),

        aspectmode="cube"
    )
)

fig.show()